# Лабораторная работа V. Нейросетевая классификация тональности отзывов Кинопоиска

**Датасет:** [Kinopoisk Movie Reviews](https://www.kaggle.com/datasets/mikhailklemin/kinopoisks-movies-reviews) (Kaggle, папки `pos` / `neg` / `neu`).

До начала обучения структура загрузки, EDA и предобработки повторяет лабораторную IV. Отличия внесены только там, где задание лабораторной V требует нейросетевой pipeline: вместо TF-IDF и классических моделей используются токенизация, padding, обучаемый `Embedding`, Conv1D, GRU и комбинированная архитектура.

## Оглавление

1. [Подготовка окружения](#Подготовка-окружения)
2. [Загрузка данных](#Загрузка-данных)
3. [Разведочный анализ данных](#Разведочный-анализ-данных)
4. [Предобработка текстов](#Предобработка-текстов)
5. [Подготовка данных к обучению](#Подготовка-данных-к-обучению)
6. [Классификация нейронными сетями](#Классификация-нейронными-сетями)
7. [Обучение и оценка моделей](#Обучение-и-оценка-моделей)
8. [Анализ ошибок](#Анализ-ошибок)
9. [Тест на новых текстах](#Тест-на-новых-текстах)
10. [Выводы](#Выводы)

## Подготовка окружения

Первый блок фиксирует воспроизводимость, подключает библиотеки для анализа данных, визуализации, лемматизации и обучения нейронных сетей. Фиксация `SEED=42` обеспечивает одинаковые разбиения и сопоставимые результаты при повторном запуске.

### 1.1 Импорты и глобальные настройки

Сначала подключаем библиотеки и проверяем обязательные зависимости. По сравнению с лабораторной IV из окружения убраны классические ML-модели, а вместо них добавлены PyTorch-компоненты для `Embedding`, Conv1D/GRU и `DataLoader`.

In [ ]:
from __future__ import annotations

import hashlib
import json
import math
import os
import platform
import random
import re
import shutil
import tempfile
import time
import warnings
from collections import Counter
from contextlib import nullcontext
from copy import deepcopy
from pathlib import Path
from typing import Iterable

MPLCONFIGDIR = Path(tempfile.gettempdir()) / "ml_lab_matplotlib_cache"
MPLCONFIGDIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPLCONFIGDIR))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import Markdown, display
from plotly.subplots import make_subplots
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
)
from sklearn.model_selection import train_test_split
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from tqdm import tqdm
from wordcloud import WordCloud

try:
    import spacy
    from spacy.lang.ru.stop_words import STOP_WORDS as SPACY_RU_STOPWORDS
except ImportError as exc:
    raise RuntimeError("spaCy не установлен. Выполните: %pip install -q spacy") from exc

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="Set2")

### 1.1.1 Воспроизводимость и служебные настройки

Фиксируем случайность, задаём метки классов и основные гиперпараметры нейросетевого эксперимента.

In [ ]:
SEED = 42
LABEL_ORDER = ["negative", "neutral", "positive"]
LABEL_TO_ID = {label: idx for idx, label in enumerate(LABEL_ORDER)}
ID_TO_LABEL = {idx: label for label, idx in LABEL_TO_ID.items()}
NUM_CLASSES = len(LABEL_ORDER)

TEXT_COL = "review"
TARGET_COL = "sentiment"

PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"
PAD_IDX = 0
UNK_IDX = 1
MIN_FREQ = 2
MAX_VOCAB_SIZE = 100_000
EMBEDDING_DIM = 200

CNN_MAX_LEN = 768
RNN_MAX_LEN = 640
COMBINED_MAX_LEN = 768
CNN_BATCH_SIZE = 512
RNN_BATCH_SIZE = 256
COMBINED_BATCH_SIZE = 256

MAX_EPOCHS = 25
PATIENCE = 5
BALANCE_STRATEGY = "sampler"  # "sampler" или "loss_weights"; одновременно не используются
CLASS_WEIGHT_ALPHA = 0.25
LABEL_SMOOTHING = 0.02
WEIGHT_DECAY = 1e-4
GRAD_CLIP = 1.0

RUN_SEED_ENSEMBLE = False
ENSEMBLE_SEEDS = (42, 77, 2026)


def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True


set_seed(SEED)
print("Imports OK")

### 1.2 Переключатель CPU/CUDA для обучения

Сначала задаём желаемый режим устройства, затем проверяем CUDA/MPS и выбираем фактическое устройство PyTorch. `DEVICE_MODE = "auto"` остаётся безопасным режимом: на CUDA-машине модель уйдёт на GPU, на Apple Silicon — на MPS, а в обычном окружении останется CPU.

In [ ]:
DEVICE_MODE = "auto"  # варианты: "auto", "cpu", "cuda", "mps"

### 1.2.1 Проверка доступности CUDA/MPS

Helper-функции аккуратно проверяют PyTorch-устройства и переводят выбранный режим в `torch.device`.

In [ ]:
def get_torch_device_status() -> dict[str, object]:
    cuda_available = bool(torch.cuda.is_available())
    mps_available = bool(getattr(torch.backends, "mps", None) and torch.backends.mps.is_available())
    return {
        "torch_version": getattr(torch, "__version__", "unknown"),
        "torch_cuda_version": getattr(getattr(torch, "version", None), "cuda", None),
        "cuda_available": cuda_available,
        "mps_available": mps_available,
    }


def resolve_training_device(mode: str = DEVICE_MODE) -> torch.device:
    normalized_mode = str(mode).strip().lower()
    allowed_modes = {"auto", "cpu", "cuda", "mps"}
    if normalized_mode not in allowed_modes:
        raise ValueError(f"DEVICE_MODE должен быть одним из {sorted(allowed_modes)}, получено: {mode!r}")

    status = get_torch_device_status()
    if normalized_mode == "cuda":
        if not status["cuda_available"]:
            raise RuntimeError("Выбран DEVICE_MODE='cuda', но CUDA недоступна в текущем окружении.")
        return torch.device("cuda")
    if normalized_mode == "mps":
        if not status["mps_available"]:
            raise RuntimeError("Выбран DEVICE_MODE='mps', но MPS недоступен в текущем окружении.")
        return torch.device("mps")
    if normalized_mode == "cpu":
        return torch.device("cpu")
    if status["cuda_available"]:
        return torch.device("cuda")
    if status["mps_available"]:
        return torch.device("mps")
    return torch.device("cpu")

### 1.2.2 Итоговое устройство обучения

Разрешаем выбранный режим в `DEVICE`, включаем AMP только на CUDA и печатаем, где фактически будут обучаться нейросетевые модели.

In [ ]:
DEVICE = resolve_training_device()
USE_AMP = DEVICE.type == "cuda"
PIN_MEMORY = DEVICE.type == "cuda"
TORCH_DEVICE_STATUS = get_torch_device_status()

print("Requested device mode:", DEVICE_MODE)
print("Torch status:", TORCH_DEVICE_STATUS)
print("Training device:", DEVICE)
print("AMP enabled:", USE_AMP)

### 1.3 Функции визуализации Plotly

Визуализации разбиты на маленькие группы: общая тема и цвета, EDA-графики, частотные графики и матрица ошибок. Так дальше в ноутбуке остаются только короткие вызовы готовых функций.


In [ ]:
PLOTLY_TEMPLATE = "plotly_white"
PLOTLY_COLORS = px.colors.qualitative.Set2

BASIC_SW = {
    "и", "в", "на", "что", "это", "как", "с", "из", "а", "но",
    "по", "за", "то", "так", "же", "был", "была", "всё", "он",
    "она", "они", "или", "к", "у", "от", "для", "его", "её",
    "их", "ещё", "уже", "о", "об", "при", "до", "со", "чем",
    "во", "бы", "ли", "тут", "тоже", "всех", "этот", "этого",
}


def show_plotly_figure(fig, title: str | None = None, *, height: int = 480):
    '''Apply the shared notebook style and render a Plotly figure.'''
    fig.update_layout(
        template=PLOTLY_TEMPLATE,
        title=title,
        title_x=0.02,
        height=height,
        margin=dict(l=40, r=30, t=75 if title else 45, b=40),
        legend_title_text="Класс",
        font=dict(size=12),
    )
    fig.show()


def make_class_color_map(labels) -> dict:
    labels = [str(label) for label in labels]
    return {label: PLOTLY_COLORS[i % len(PLOTLY_COLORS)] for i, label in enumerate(labels)}


### 1.3.1 Распределения классов и длин

Функция строит пару графиков для EDA: число отзывов по классам и распределение длин текстов.


In [ ]:
def plot_class_counts_and_lengths(
    data: pd.DataFrame,
    *,
    target_col: str = "sentiment",
    word_count_col: str = "word_count",
):
    counts = data[target_col].value_counts()
    color_map = make_class_color_map(counts.index)

    fig = make_subplots(
        rows=1,
        cols=2,
        subplot_titles=("Распределение классов", "Распределение длин отзывов"),
    )
    fig.add_trace(
        go.Bar(
            x=counts.index.astype(str),
            y=counts.values,
            text=[f"{value:,}" for value in counts.values],
            textposition="outside",
            marker_color=[color_map[str(label)] for label in counts.index],
            name="Количество отзывов",
            showlegend=False,
        ),
        row=1,
        col=1,
    )

    for label in counts.index:
        subset = data.loc[data[target_col] == label, word_count_col]
        fig.add_trace(
            go.Histogram(
                x=subset,
                nbinsx=60,
                opacity=0.55,
                marker_color=color_map[str(label)],
                name=str(label),
            ),
            row=1,
            col=2,
        )

    fig.update_layout(barmode="overlay")
    fig.update_xaxes(title_text="Класс", row=1, col=1)
    fig.update_yaxes(title_text="Количество отзывов", row=1, col=1)
    fig.update_xaxes(title_text="Количество слов", range=[0, 500], row=1, col=2)
    fig.update_yaxes(title_text="Частота", row=1, col=2)
    return show_plotly_figure(fig, "Базовая статистика датасета", height=500)


### 1.3.2 WordCloud и топ слов

Здесь собраны функции для облаков слов и топов частот. Это ещё сырой текст, поэтому стоп-слова фильтруются только грубо.


In [ ]:
def plot_wordclouds_by_class(
    data: pd.DataFrame,
    *,
    target_col: str = "sentiment",
    text_col: str = "review",
    max_words: int = 150,
):
    classes = data[target_col].dropna().unique().tolist()
    cmaps = ["Blues", "Reds", "Greens", "Purples", "Oranges", "Greys"]
    fig = make_subplots(
        rows=1,
        cols=len(classes),
        subplot_titles=[f"Облако слов — {str(label).capitalize()}" for label in classes],
    )

    for col_idx, label in enumerate(classes, start=1):
        text = " ".join(data.loc[data[target_col] == label, text_col].astype(str))
        wc = WordCloud(
            width=700,
            height=420,
            background_color="white",
            colormap=cmaps[(col_idx - 1) % len(cmaps)],
            max_words=max_words,
            collocations=False,
        ).generate(text)
        fig.add_trace(go.Image(z=wc.to_array(), hoverinfo="skip"), row=1, col=col_idx)
        fig.update_xaxes(visible=False, row=1, col=col_idx)
        fig.update_yaxes(visible=False, row=1, col=col_idx)

    return show_plotly_figure(
        fig,
        "Наиболее частотные слова по классам (до предобработки)",
        height=500,
    )


def get_top_words(texts, *, n: int = 20, stopwords: set[str] | None = None):
    stopwords = BASIC_SW if stopwords is None else stopwords
    words = [
        word.lower()
        for text in texts
        for word in re.findall(r"[а-яёА-ЯЁ]+", str(text))
        if word.lower() not in stopwords and len(word) > 2
    ]
    return Counter(words).most_common(n)


def plot_top_words_by_class(
    data: pd.DataFrame,
    *,
    target_col: str = "sentiment",
    text_col: str = "review",
    n: int = 20,
):
    classes = data[target_col].dropna().unique().tolist()
    color_map = make_class_color_map(classes)
    fig = make_subplots(
        rows=1,
        cols=len(classes),
        subplot_titles=[f"Топ-{n} слов — {str(label).capitalize()}" for label in classes],
    )

    for col_idx, label in enumerate(classes, start=1):
        top = get_top_words(data.loc[data[target_col] == label, text_col], n=n)
        if not top:
            continue
        words, freqs = zip(*top)
        fig.add_trace(
            go.Bar(
                x=list(freqs)[::-1],
                y=list(words)[::-1],
                orientation="h",
                marker_color=color_map[str(label)],
                name=str(label),
                showlegend=False,
            ),
            row=1,
            col=col_idx,
        )
        fig.update_xaxes(title_text="Частота", row=1, col=col_idx)

    return show_plotly_figure(fig, f"Топ-{n} слов по классам", height=580)


### 1.3.3 Матрица ошибок

Отдельная функция рисует confusion matrix в едином Plotly-оформлении.


In [ ]:
def plot_confusion_matrix_plotly(
    y_true,
    y_pred,
    labels,
    *,
    title: str = "Матрица ошибок",
):
    matrix = confusion_matrix(y_true, y_pred, labels=labels)
    matrix_df = pd.DataFrame(matrix, index=labels, columns=labels)
    fig = px.imshow(
        matrix_df,
        text_auto=True,
        color_continuous_scale="Blues",
        aspect="auto",
        labels=dict(x="Предсказано", y="Истинный класс", color="Количество"),
    )
    fig.update_traces(hovertemplate="Истинный класс: %{y}<br>Предсказано: %{x}<br>Количество: %{z}<extra></extra>")
    return show_plotly_figure(fig, title, height=520)


### 1.4 Загрузка языковой модели spaCy

`ru_core_news_sm` — компактная модель для русского языка. Если она установлена, используется лемматизация как в лабораторной IV. Если модель отсутствует в окружении, notebook не падает: включается `spacy.blank("ru")`, а предобработка работает как мягкая токенизация без лемматизации.

Для полного повторения Lab IV можно установить модель отдельно:
```
python -m spacy download ru_core_news_sm
```


In [ ]:
SPACY_MODEL = "ru_core_news_sm"
SPACY_HAS_LEMMATIZER = False

try:
    nlp = spacy.load(SPACY_MODEL, disable=["parser", "ner"])
    SPACY_HAS_LEMMATIZER = "lemmatizer" in nlp.pipe_names
    print("spaCy model loaded:", nlp.meta.get("name"), nlp.meta.get("version"))
except OSError:
    nlp = spacy.blank("ru")
    print(
        f"spaCy model {SPACY_MODEL!r} не найдена; используется spacy.blank('ru') "
        "без лемматизации. Для полного повторения Lab IV установите ru_core_news_sm."
    )


Окружение готово. Если `ru_core_news_sm` доступна, предобработка использует лемматизацию Lab IV; если нет — notebook остаётся исполнимым и использует мягкую токенизацию spaCy без лемматизации.


## Загрузка данных

Датасет Kinopoisk Movie Reviews с Kaggle хранится не как один CSV-файл:
в архиве лежат директории классов `pos`, `neg`, `neu`, а каждый отзыв находится
в отдельном `.txt`-файле. В этом разделе приводим такой формат к рабочей таблице
с колонками `review` и `sentiment`.


### 2.1 Подготовка Kaggle-датасета и первичный осмотр

Блок разбит на четыре шага: пути и константы, поиск исходных папок `pos`/`neg`/`neu`, чтение `.txt`-файлов и финальная загрузка prepared-таблицы из кэша или пересборка. Повторные запуски не перечитывают датасет без причины.


In [ ]:
PREPARED_DATA_FILENAME = "kinopoisk_reviews_prepared.csv"
KAGGLE_DATASET = "mikhailklemin/kinopoisks-movies-reviews"
LABEL_DIR_MAP = {
    "neg": "negative",
    "neu": "neutral",
    "pos": "positive",
}


def find_lab_iv_dir() -> Path:
    cwd = Path.cwd().resolve()
    for base in [cwd, *cwd.parents]:
        if base.name == "LAB_IV" and (base / "data").exists():
            return base
        lab_candidate = base / "LAB_IV"
        if (lab_candidate / "data").exists():
            return lab_candidate
    raise FileNotFoundError("Не найдена директория LAB_IV/data относительно текущего пути запуска")


LAB_IV_DIR = find_lab_iv_dir()
DATA_DIR = LAB_IV_DIR / "data"
DATASET_DIR = DATA_DIR / "dataset"
DATASET_DIR.mkdir(parents=True, exist_ok=True)
DATA_PATH = DATASET_DIR / PREPARED_DATA_FILENAME


### 2.1.1 Поиск и подготовка raw-датасета

Helper-функции находят папки классов, при необходимости скачивают Kaggle-датасет и приводят локальную структуру к одному виду.


In [ ]:
def count_txt_files(root: Path) -> int:
    return sum(1 for _ in root.rglob("*.txt")) if root.exists() else 0


def get_review_dir_counts(root: Path) -> dict[str, int]:
    return {
        label_dir: count_txt_files(root / label_dir)
        for label_dir in LABEL_DIR_MAP
    }


def has_review_dirs(root: Path) -> bool:
    return all((root / label_dir).is_dir() for label_dir in LABEL_DIR_MAP)


def find_review_dirs_root(root: Path) -> Path | None:
    root = Path(root)
    if has_review_dirs(root):
        return root
    if root.exists():
        for candidate in root.rglob("*"):
            if candidate.is_dir() and has_review_dirs(candidate):
                return candidate
    return None


def copy_review_dirs(source_root: Path, target_root: Path) -> None:
    source_root = source_root.resolve()
    target_root = target_root.resolve()
    if source_root == target_root:
        return

    target_root.mkdir(parents=True, exist_ok=True)
    for label_dir in LABEL_DIR_MAP:
        src = source_root / label_dir
        dst = target_root / label_dir
        dst.mkdir(parents=True, exist_ok=True)
        for src_file in src.rglob("*.txt"):
            rel_path = src_file.relative_to(src)
            dst_file = dst / rel_path
            dst_file.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src_file, dst_file)


def download_kaggle_dataset() -> Path:
    try:
        import kagglehub
    except ImportError as exc:
        raise RuntimeError(
            "Папки pos/neg/neu не найдены и не установлен kagglehub. "
            "Установите зависимости через uv sync --dev или добавьте kagglehub вручную."
        ) from exc

    errors: list[str] = []
    download_attempts = [{"output_dir": str(DATASET_DIR)}, {}]
    for kwargs in download_attempts:
        try:
            return Path(kagglehub.dataset_download(KAGGLE_DATASET, **kwargs))
        except TypeError as exc:
            errors.append(f"dataset_download({kwargs}) не поддерживается: {exc}")
        except Exception as exc:
            errors.append(f"dataset_download({kwargs}) завершился ошибкой: {exc}")

    raise RuntimeError("Не удалось скачать Kaggle-датасет. Ошибки: " + " | ".join(errors))


def ensure_raw_dataset() -> Path:
    local_root = find_review_dirs_root(DATASET_DIR)
    if local_root is not None:
        copy_review_dirs(local_root, DATASET_DIR)
        return DATASET_DIR

    downloaded_dir = download_kaggle_dataset()
    downloaded_root = find_review_dirs_root(downloaded_dir)
    if downloaded_root is None:
        raise FileNotFoundError(
            "В скачанном Kaggle-датасете не найдены папки pos/neg/neu. "
            f"Проверенный путь: {downloaded_dir.resolve()}"
        )

    copy_review_dirs(downloaded_root, DATASET_DIR)
    return DATASET_DIR


### 2.1.2 Сборка таблицы из `.txt`

Каждый файл читается с запасными кодировками, из имени достаются `movie_id` и `review_id`, а затем всё собирается в один `DataFrame`.


In [ ]:
def read_review_text(path: Path) -> str:
    for encoding in ("utf-8-sig", "utf-8", "cp1251"):
        try:
            return path.read_text(encoding=encoding).strip()
        except UnicodeDecodeError:
            continue
    return path.read_text(encoding="utf-8", errors="replace").strip()


def parse_review_ids(path: Path) -> tuple[int | None, int | None]:
    parts = path.stem.split("-", maxsplit=1)
    if len(parts) != 2:
        return None, None
    try:
        return int(parts[0]), int(parts[1])
    except ValueError:
        return None, None


def build_reviews_dataframe(raw_root: Path) -> pd.DataFrame:
    records = []
    total_files = sum(get_review_dir_counts(raw_root).values())
    progress = tqdm(
        total=total_files,
        desc="Reading review txt files",
        dynamic_ncols=True,
        mininterval=1.0,
        leave=False,
    )

    with progress:
        for label_dir, sentiment in LABEL_DIR_MAP.items():
            label_root = raw_root / label_dir
            for txt_path in sorted(label_root.rglob("*.txt")):
                movie_id, review_id = parse_review_ids(txt_path)
                records.append({
                    "review": read_review_text(txt_path),
                    "sentiment": sentiment,
                    "source_label_dir": label_dir,
                    "source_file": str(txt_path.relative_to(raw_root)),
                    "movie_id": movie_id,
                    "review_id": review_id,
                })
                progress.update(1)

    if not records:
        raise ValueError(f"В {raw_root.resolve()} не найдено ни одного .txt-отзыва")

    result = pd.DataFrame.from_records(records)
    result["movie_id"] = result["movie_id"].astype("Int64")
    result["review_id"] = result["review_id"].astype("Int64")
    return result


### 2.1.3 Prepared-кэш

Если `kinopoisk_reviews_prepared.csv` совпадает с raw-датасетом по размеру и колонкам, берём его. Иначе пересобираем таблицу заново.


In [ ]:
def load_or_build_reviews_dataframe() -> tuple[pd.DataFrame, Path, dict[str, int]]:
    raw_dataset_root = ensure_raw_dataset()
    raw_counts = get_review_dir_counts(raw_dataset_root)
    raw_total = sum(raw_counts.values())

    if raw_total == 0:
        raise ValueError(f"Папки pos/neg/neu найдены в {raw_dataset_root.resolve()}, но .txt-файлов в них нет")

    cache_is_fresh = False
    if DATA_PATH.exists():
        cached_df = pd.read_csv(DATA_PATH)
        has_required_columns = {"review", "sentiment"}.issubset(cached_df.columns)
        cache_is_fresh = has_required_columns and len(cached_df) == raw_total
        if cache_is_fresh:
            df = cached_df
            print("Loaded prepared table:", DATA_PATH.resolve())
        else:
            print("Prepared table exists, but does not match raw dataset. Rebuilding cache...")

    if not cache_is_fresh:
        df = build_reviews_dataframe(raw_dataset_root)
        df.to_csv(DATA_PATH, index=False)
        print("Prepared table saved:", DATA_PATH.resolve())

    return df, raw_dataset_root, raw_counts


df, raw_dataset_root, raw_counts = load_or_build_reviews_dataframe()

print("Dataset directory:", DATASET_DIR.resolve())
print("Raw review counts:", raw_counts)
print("Prepared table:", DATA_PATH.resolve())
print("Prepared shape:", df.shape)


### 2.1.4 Быстрый осмотр таблицы

Печатаем путь к prepared-файлу, первые строки и распределение классов, чтобы сразу проверить форму данных глазами.


In [ ]:
print("Prepared data path:", DATA_PATH.resolve())
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head())
df.info()


### 2.2 Нормализация столбцов и меток

После сборки из `.txt`-файлов таблица уже содержит `review` и `sentiment`.
Дополнительно нормализуем метки классов, удаляем строки с пустым текстом или
отсутствующей меткой и сохраняем служебные поля `source_file`, `movie_id`,
`review_id` для последующего анализа ошибок.


In [ ]:
required_columns = {"review", "sentiment"}
missing_columns = required_columns.difference(df.columns)
if missing_columns:
    raise ValueError(f"В подготовленной таблице нет обязательных колонок: {sorted(missing_columns)}")

metadata_columns = [
    column
    for column in ["source_label_dir", "source_file", "movie_id", "review_id"]
    if column in df.columns
]

df = df[["review", "sentiment", *metadata_columns]].copy()

# Normalize sentiment labels to lowercase strings
label_map = {
    "1": "positive", "0": "negative", "-1": "negative",
    "good": "positive", "bad": "negative",
    "pos": "positive", "neg": "negative", "neu": "neutral",
    "positive": "positive", "negative": "negative", "neutral": "neutral",
    "позитивный": "positive", "негативный": "negative", "нейтральный": "neutral",
}

df["sentiment"] = (
    df["sentiment"]
    .astype(str)
    .str.strip()
    .str.lower()
    .replace(label_map)
)

# Drop nulls and empty reviews
df.dropna(subset=["review", "sentiment"], inplace=True)
df["review"] = df["review"].astype(str)
df = df[df["review"].str.strip().astype(bool)].reset_index(drop=True)

expected_labels = set(LABEL_DIR_MAP.values())
unexpected_labels = sorted(set(df["sentiment"]) - expected_labels)
if unexpected_labels:
    print("Warning: unexpected labels after normalization:", unexpected_labels)

print("Shape after cleaning:", df.shape)
print("Unique sentiments:", sorted(df["sentiment"].unique()))
print(df["sentiment"].value_counts())


> **Вывод:**

Датасет загрузился без потерь: всего прочитано `131669` `.txt`-отзывов, из них `87138` из `pos`, `24704` из `neu` и `19827` из `neg`. После нормализации таблица сохранила размер `(131669, 6)`, все основные поля заполнены, а пустые отзывы после очистки не появились.

Распределение классов сразу показывает главную особенность задачи: `positive` сильно доминирует над `neutral` и `negative`. В целом данные пригодны для классификации тональности, но дальше нельзя ориентироваться только на `accuracy`, потому что модель может неплохо выглядеть просто за счет частого положительного класса.


## Разведочный анализ данных

Цель EDA — понять распределение классов, характер текстов и наиболее
информативные слова каждого класса. Суммарно не более 8 графиков.


### 3.1 Распределение классов и длины текстов

Два графика в одной строке:
столбчатая диаграмма числа отзывов по классам
и гистограмма длин текстов (в словах) с разбивкой по классам.
Это показывает, есть ли дисбаланс классов
и насколько тексты однородны по длине.


In [ ]:
df["word_count"] = df["review"].str.split().str.len()
plot_class_counts_and_lengths(df)

print(df.groupby("sentiment")["word_count"].describe().round(1))


> **Вывод:**

Самый частый класс — `positive` (`87138` отзывов), самый редкий — `negative` (`19827` отзывов). Разница между ними примерно в `4.4` раза, поэтому `accuracy` будет довольно снисходительной метрикой: можно часто угадывать позитив и получать неплохое качество, но при этом плохо работать с редкими классами. Поэтому `macro_f1` здесь полезнее, он честнее наказывает за слабые `neutral` и `negative`.

По длинам классы похожи: средняя длина держится около `329-345` слов, медиана около `285-298`. При этом есть длинные выбросы до `2059` слов, но это скорее особенность отзывов Кинопоиска, а не ошибка данных. Для обучения я считаю правильным использовать `class_weight` / `auto_class_weights`, иначе модель почти неизбежно будет тянуться к `positive`.


### 3.2 Облака слов по классам

WordCloud визуализирует наиболее частотные слова каждого класса на сырых текстах
(до предобработки). Это даёт интуитивное представление о лексиконе классов.


In [ ]:
plot_wordclouds_by_class(df)


Главный практический вывод такой: лексический сигнал есть, но на облаках он забит слишком общими словами. Без нормальной предобработки и контролируемого словаря нейросеть будет видеть не столько тональность, сколько сам факт того, что это отзыв о фильме.

### 3.3 Топ-20 слов по классам

Гистограмма частот 20 самых популярных слов для каждого класса
после грубой фильтрации служебных частей речи через `Counter`.
Это количественная картина поверх облаков слов.

> Слово «не» **намеренно оставлено** в обоих наборах —
> оно несёт семантическую нагрузку в негативных отзывах.


In [ ]:
plot_top_words_by_class(df)


> **Вывод:**

Топ-20 подтверждает картину из облаков: во всех классах наверху стоят `фильм`, `все`, `фильма`, `очень`, `просто`, `если`, `который`, `кино`. В `negative` заметно слово `нет` (`16478` вхождений), а в `positive` особенно часто встречается `очень` (`108476`), но в целом пересечение топов слишком большое, чтобы по ним одним отделить тональность.

Частицу `не` и слово `нет` важно не выкидывать: для отзывов это не просто служебные слова, а переключатели смысла. Если удалить отрицание, фразы вроде `не понравилось` или `не отвечает требованиям` могут стать почти положительными по набору оставшихся слов, и это как раз тот случай, где простая очистка может испортить модель.


### 3.4 Общая статистика словаря

Считаем суммарный объём словаря, среднюю и медианную длину отзыва.
Эта числовая сводка дополняет визуализации.


In [ ]:
all_words = [
    w.lower() for text in df["review"].astype(str)
    for w in re.findall(r"[а-яёА-ЯЁ]+", text)
]
print(f"Всего токенов:      {len(all_words):>10,}")
print(f"Уникальных токенов: {len(set(all_words)):>10,}")
print(f"\nСредняя длина отзыва: {df['word_count'].mean():.1f} слов")
print(f"Медиана длины:        {df['word_count'].median():.1f} слов")
print(f"Максимальная длина:   {df['word_count'].max()} слов")


Для нейросетевого токенизатора нужен ограниченный словарь: полный уникальный словарь слишком велик, а длинный хвост редких слов почти не помогает обучаемому `Embedding`. Поэтому дальше словарь строится только по train-части с ограничением `MAX_VOCAB_SIZE` и минимальной частотой токена.

## Предобработка текстов

Предобработка приводит тексты к нормализованному виду:
нижний регистр → удаление пунктуации и цифр → лемматизация spaCy →
удаление стоп-слов (кроме «не» и «нет»).
Результат сохраняется в столбце `cleaned_text`.


### 4.1 Набор стоп-слов

Используем встроенный список spaCy для русского языка,
но исключаем «не» и «нет» — они принципиально меняют смысл высказывания
(ср. «хороший» vs «не хороший»).


In [ ]:
STOPWORDS = set(SPACY_RU_STOPWORDS)
STOPWORDS.discard("не")
STOPWORDS.discard("нет")

print(f"Стоп-слов в наборе: {len(STOPWORDS)}")
print(f"'не' в стоп-словах: {'не' in STOPWORDS}")
print(f"'нет' в стоп-словах: {'нет' in STOPWORDS}")


### 4.2 Функция предобработки одного текста

Шаги:
1. Нижний регистр.
2. Удаление пунктуации и цифр (оставляем только кириллицу и пробелы).
3. Лемматизация через spaCy: каждый токен заменяется нормальной формой.
4. Удаление стоп-слов и коротких токенов (len < 2).


In [ ]:
def normalize_text(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r"[^а-яёa-z\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def token_normal_form(token) -> str:
    lemma = getattr(token, "lemma_", "")
    value = lemma if SPACY_HAS_LEMMATIZER and lemma else token.text
    return value.lower().strip()


def clean_doc(doc) -> str:
    normalized_tokens = [token_normal_form(token) for token in doc if not token.is_space]
    return " ".join(
        token for token in normalized_tokens
        if token not in STOPWORDS
        and len(token) >= 2
    )


def preprocess_text(text: str) -> str:
    return clean_doc(nlp(normalize_text(text)))


def preprocess_texts(texts, batch_size: int = 256, n_process: int = 1) -> list[str]:
    normalized_texts = [normalize_text(text) for text in texts]
    docs = nlp.pipe(normalized_texts, batch_size=batch_size, n_process=n_process)
    return [
        clean_doc(doc)
        for doc in tqdm(
            docs,
            total=len(normalized_texts),
            desc="Preprocessing",
            dynamic_ncols=True,
            mininterval=1.0,
            leave=False,
        )
    ]


# Smoke test
sample = "Этот фильм не оставил меня равнодушным! Просто великолепно."
print("Вход: ", sample)
print("Выход:", preprocess_text(sample))

> **Вывод:**

Smoke-test отработал так, как и хотелось: фраза `Этот фильм не оставил меня равнодушным! Просто великолепно.` превратилась в `фильм не оставить равнодушный великолепно`. Частица `не` сохранилась, пунктуация и лишние символы ушли, а слова стали ближе к нормальной форме: `оставил` → `оставить`, `равнодушным` → `равнодушный`.

Перед обучением на полном датасете важно было проверить, не появляются ли пустые `cleaned_text` и не слишком ли агрессивно стоп-слова вычищают смысл. Особенно это касается отрицаний, потому что для тональности они важнее многих обычных частотных слов.


### 4.3 Применение к датасету

Предобработка вынесена в кэшируемый блок: отдельно задаём параметры и пути, отдельно проверяем пригодность кэша, а в финальной ячейке либо загружаем `cleaned_text`, либо запускаем долгий проход spaCy.


In [ ]:
PREPROCESS_BATCH_SIZE = 256
PREPROCESS_N_PROCESS = min(4, os.cpu_count() or 1)
PREPROCESSING_VERSION = "spacy-lemma-v1"

CLEANED_PARQUET_PATH = DATA_PATH.with_name("kinopoisk_reviews_cleaned.parquet")
CLEANED_CSV_PATH = DATA_PATH.with_name("kinopoisk_reviews_cleaned.csv")
CACHE_META_PATH = DATA_PATH.with_name("kinopoisk_reviews_cleaned.meta.json")
CACHE_META_KEYS = (
    "spacy_model",
    "spacy_model_version",
    "preprocessing_version",
    "source_rows",
    "source_size_bytes",
    "review_chars_total",
    "stopwords",
)


### 4.3.1 Проверка кэша предобработки

Метаданные фиксируют версию spaCy, размер исходных данных и набор стоп-слов. Это защищает от ситуации, когда старый `cleaned_text` случайно используется после изменения очистки.


In [ ]:
def build_cache_metadata(source_df: pd.DataFrame) -> dict:
    return {
        "spacy_model": SPACY_MODEL,
        "spacy_model_version": nlp.meta.get("version"),
        "preprocessing_version": PREPROCESSING_VERSION,
        "source_rows": int(len(source_df)),
        "source_path": str(DATA_PATH.resolve()),
        "source_size_bytes": int(DATA_PATH.stat().st_size) if DATA_PATH.exists() else None,
        "review_chars_total": int(source_df["review"].fillna("").str.len().sum()),
        "stopwords": sorted(STOPWORDS),
    }


def cache_frame_is_usable(cached_df: pd.DataFrame, source_df: pd.DataFrame) -> bool:
    if "cleaned_text" not in cached_df.columns:
        return False
    if len(cached_df) != len(source_df):
        return False

    required_columns = [column for column in ("review", "sentiment") if column in source_df.columns]
    return all(column in cached_df.columns for column in required_columns)


def cache_metadata_matches(expected_meta: dict) -> bool:
    if not CACHE_META_PATH.exists():
        return False

    try:
        with CACHE_META_PATH.open("r", encoding="utf-8") as cache_meta_file:
            actual_meta = json.load(cache_meta_file)
    except (OSError, json.JSONDecodeError):
        return False

    return all(actual_meta.get(key) == expected_meta.get(key) for key in CACHE_META_KEYS)


def write_cache_metadata(meta: dict) -> None:
    with CACHE_META_PATH.open("w", encoding="utf-8") as cache_meta_file:
        json.dump(meta, cache_meta_file, ensure_ascii=False, indent=2)


def read_cleaned_cache(cache_path: Path) -> pd.DataFrame:
    if cache_path.suffix == ".parquet":
        return pd.read_parquet(cache_path)
    return pd.read_csv(cache_path)


def find_cleaned_cache(source_df: pd.DataFrame, expected_meta: dict):
    for cache_path in (CLEANED_PARQUET_PATH, CLEANED_CSV_PATH):
        if not cache_path.exists():
            continue

        print(f"Found cleaned cache: {cache_path.resolve()}")
        cached_df = read_cleaned_cache(cache_path)
        if not cache_frame_is_usable(cached_df, source_df):
            print("Cache schema or row count does not match. Rebuilding...")
            continue

        if cache_metadata_matches(expected_meta):
            return cached_df, cache_path

        if not CACHE_META_PATH.exists():
            print("Cache metadata is missing; using cache after basic shape check.")
            write_cache_metadata(expected_meta)
            return cached_df, cache_path

        print("Cache metadata does not match current preprocessing. Rebuilding...")

    return None, None


### 4.3.2 Загрузка или пересчёт `cleaned_text`

Если кэш валиден, берём его. Если нет — обрабатываем отзывы через `nlp.pipe`, сохраняем Parquet и показываем несколько примеров.


In [ ]:
def load_or_preprocess_reviews(source_df: pd.DataFrame) -> pd.DataFrame:
    expected_cache_meta = build_cache_metadata(source_df)
    cached_df, cache_path = find_cleaned_cache(source_df, expected_cache_meta)

    if cached_df is not None:
        print(f"Loaded cleaned data from cache → {cache_path.resolve()}")
        if cache_path != CLEANED_PARQUET_PATH:
            cached_df.to_parquet(CLEANED_PARQUET_PATH, index=False)
            write_cache_metadata(expected_cache_meta)
            print(f"Parquet cache saved → {CLEANED_PARQUET_PATH.resolve()}")
        return cached_df

    prepared_df = source_df.copy()
    prepared_df["cleaned_text"] = preprocess_texts(
        prepared_df["review"].fillna("").tolist(),
        batch_size=PREPROCESS_BATCH_SIZE,
        n_process=PREPROCESS_N_PROCESS,
    )
    prepared_df.to_parquet(CLEANED_PARQUET_PATH, index=False)
    write_cache_metadata(expected_cache_meta)
    print(f"Cleaned data saved → {CLEANED_PARQUET_PATH.resolve()}")
    return prepared_df


df = load_or_preprocess_reviews(df)

print("\nПримеры предобработки:")
for _, row in df.sample(3, random_state=SEED).iterrows():
    print("ORIGINAL:", str(row["review"])[:120])
    print("CLEANED: ", str(row["cleaned_text"])[:120])
    print()


### 4.4 Проверка качества предобработки

Проверяем, что предобработка не породила пустых строк,
и смотрим на по одному примеру каждого класса.


In [ ]:
empty_mask = df["cleaned_text"].str.strip().str.len() == 0
print(f"Пустых cleaned_text: {empty_mask.sum()}")
df = df[~empty_mask].reset_index(drop=True)

for cls in df["sentiment"].unique():
    print(f"\n--- {cls.upper()} ---")
    row = df[df["sentiment"] == cls].sample(1, random_state=SEED).iloc[0]
    print("ORIGINAL:", str(row["review"])[:150])
    print("CLEANED: ", str(row["cleaned_text"])[:150])


Признаков слишком агрессивной очистки я не вижу. Да, часть форм выглядит немного шероховато из-за spaCy и в тексте остаются английские названия вроде `the runaways`, но для токенизации и обучаемого `Embedding` это допустимый шум. Стоп-слова, регулярку и параметры `nlp.pipe` в этой версии можно оставить без изменений.

## Подготовка данных к обучению

До этого места загрузка, EDA и лемматизация повторяют лабораторную IV. В этом разделе начинается отличие лабораторной V: вместо TF-IDF строим словарь только по train-части, кодируем тексты в последовательности индексов и приводим их к единой длине через padding.

### 5.1 Train / validation / test split

Стратифицированное разделение сохраняет естественное соотношение классов. Test остаётся отложенным до финальной оценки, а validation используется для early stopping и calibration.

In [ ]:
model_df = df.copy()
model_df = model_df[model_df["cleaned_text"].astype(str).str.strip().astype(bool)].reset_index(drop=True)
model_df = model_df[model_df[TARGET_COL].isin(LABEL_ORDER)].reset_index(drop=True)

train_part_df, test_df = train_test_split(
    model_df,
    test_size=0.20,
    stratify=model_df[TARGET_COL],
    random_state=SEED,
)
train_df, val_df = train_test_split(
    train_part_df,
    test_size=0.15,
    stratify=train_part_df[TARGET_COL],
    random_state=SEED,
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

for split_name, split_df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    split_df["split"] = split_name
model_df = pd.concat([train_df, val_df, test_df], ignore_index=True)

print(len(train_df), len(val_df), len(test_df))
display(
    pd.crosstab(model_df["split"], model_df[TARGET_COL], normalize="index")
    .reindex(index=["train", "val", "test"], columns=LABEL_ORDER)
    .round(3)
)

### 5.2 Vocabulary по train

Токенизатор реализован через `Counter`: это прямой аналог `Tokenizer`, но без привязки к TensorFlow/Keras. Словарь строится только по train-части, чтобы validation и test не влияли на обучение.

In [ ]:
def iter_tokens(texts: Iterable[str]):
    for text in texts:
        yield from str(text).split()


def build_vocab(texts: Iterable[str], max_vocab_size: int = MAX_VOCAB_SIZE, min_freq: int = MIN_FREQ):
    counter = Counter(iter_tokens(texts))
    tokens = [token for token, freq in counter.most_common(max_vocab_size - 2) if freq >= min_freq]
    vocab = {PAD_TOKEN: PAD_IDX, UNK_TOKEN: UNK_IDX}
    vocab.update({token: idx for idx, token in enumerate(tokens, start=2)})
    return vocab, counter


vocab, token_counter = build_vocab(train_df["cleaned_text"])
vocab_hash = hashlib.md5("\n".join(vocab.keys()).encode("utf-8")).hexdigest()[:12]

print(f"Vocab size: {len(vocab):,}")
print(f"Vocab hash: {vocab_hash}")
display(pd.DataFrame(token_counter.most_common(20), columns=["token", "count"]))

### 5.3 Обоснование `max_len`

Считаем длины очищенных train-текстов и сравниваем их с выбранными лимитами для CNN/RNN. CNN получает более длинный контекст, RNN ограничена сильнее из-за последовательной природы GRU.

In [ ]:
train_lengths = train_df["cleaned_text"].str.split().str.len()
length_stats = train_lengths.describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]).to_frame("train_tokens")
display(length_stats)
print(f"CNN_MAX_LEN={CNN_MAX_LEN}, RNN_MAX_LEN={RNN_MAX_LEN}, COMBINED_MAX_LEN={COMBINED_MAX_LEN}")

### 5.4 Кодирование текстов

`truncate_head_tail` сохраняет начало и конец отзыва. Это важнее простого обрезания по началу, потому что итоговая оценка часто формулируется в финальных предложениях.

In [ ]:
def truncate_head_tail(ids: list[int], max_len: int) -> list[int]:
    if len(ids) <= max_len:
        return ids
    head_len = max_len // 2
    tail_len = max_len - head_len
    return ids[:head_len] + ids[-tail_len:]


def encode_one(text: str, vocab: dict[str, int], max_len: int) -> tuple[list[int], int]:
    ids = [vocab.get(token, UNK_IDX) for token in str(text).split()]
    if not ids:
        ids = [UNK_IDX]
    ids = truncate_head_tail(ids, max_len)
    length = len(ids)
    padded = ids + [PAD_IDX] * (max_len - length)
    return padded, length


def encode_texts(texts: Iterable[str], vocab: dict[str, int], max_len: int) -> tuple[torch.Tensor, torch.Tensor]:
    encoded = [encode_one(text, vocab, max_len) for text in tqdm(list(texts), desc=f"encode:{max_len}")]
    input_ids = torch.tensor([row[0] for row in encoded], dtype=torch.long)
    lengths = torch.tensor([row[1] for row in encoded], dtype=torch.long)
    return input_ids, lengths

### 5.5 Encoded-cache

Закодированные тензоры сохраняются отдельно для каждого split, версии предобработки, словаря и `max_len`. Кэш складывается в `LAB_V/data/dataset`, чтобы не смешивать нейросетевые артефакты с лабораторной IV.

In [ ]:
LAB_V_DIR = LAB_IV_DIR.parent / "LAB_V"
LAB5_DATASET_DIR = LAB_V_DIR / "data" / "dataset"
LAB5_DATASET_DIR.mkdir(parents=True, exist_ok=True)
ENCODED_CACHE_DIR = LAB5_DATASET_DIR / "encoded_cache"
ENCODED_CACHE_DIR.mkdir(parents=True, exist_ok=True)


def encoded_cache_path(split: str, max_len: int) -> Path:
    safe_preprocess_version = re.sub(r"[^a-zA-Z0-9_.-]+", "_", PREPROCESSING_VERSION)
    return ENCODED_CACHE_DIR / f"encoded_{safe_preprocess_version}_{vocab_hash}_{split}_{max_len}.pt"


def labels_to_tensor(labels: Iterable[str]) -> torch.Tensor:
    return torch.tensor([LABEL_TO_ID[label] for label in labels], dtype=torch.long)


def load_torch_payload(path: Path):
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")


def load_or_encode_split(split: str, split_df: pd.DataFrame, max_len: int) -> dict[str, torch.Tensor]:
    path = encoded_cache_path(split, max_len)
    if path.exists():
        return load_torch_payload(path)

    input_ids, lengths = encode_texts(split_df["cleaned_text"], vocab, max_len)
    payload = {
        "input_ids": input_ids,
        "lengths": lengths,
        "labels": labels_to_tensor(split_df[TARGET_COL]),
        "meta": {
            "split": split,
            "max_len": max_len,
            "preprocessing_version": PREPROCESSING_VERSION,
            "vocab_hash": vocab_hash,
        },
    }
    torch.save(payload, path)
    return payload

### 5.6 Подготовка encoded tensors

Готовим разные длины последовательностей для CNN, RNN и комбинированной модели.

In [ ]:
split_frames = {"train": train_df, "val": val_df, "test": test_df}
max_lens = sorted({CNN_MAX_LEN, RNN_MAX_LEN, COMBINED_MAX_LEN})
encoded_by_max_len = {
    max_len: {split: load_or_encode_split(split, split_df, max_len) for split, split_df in split_frames.items()}
    for max_len in max_lens
}

for max_len, splits in encoded_by_max_len.items():
    print(max_len, {split: tuple(payload["input_ids"].shape) for split, payload in splits.items()})

### 5.7 Dataset и DataLoader

Dataset возвращает `input_ids`, реальные длины и метки. Для Windows `num_workers` автоматически ставится в `0`, чтобы избежать падений multiprocessing в ноутбуке.

In [ ]:
class ReviewDataset(Dataset):
    def __init__(self, encoded_split: dict[str, torch.Tensor]):
        self.input_ids = encoded_split["input_ids"]
        self.lengths = encoded_split["lengths"]
        self.labels = encoded_split["labels"]

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, idx: int) -> dict[str, torch.Tensor]:
        return {
            "input_ids": self.input_ids[idx],
            "lengths": self.lengths[idx],
            "labels": self.labels[idx],
        }


def resolve_dataloader_workers() -> int:
    if platform.system().lower().startswith("win"):
        return 0
    return min(4, os.cpu_count() or 0)


def make_weighted_sampler(labels: torch.Tensor) -> WeightedRandomSampler:
    counts = torch.bincount(labels, minlength=NUM_CLASSES).float()
    class_weights_for_sampling = labels.numel() / (NUM_CLASSES * counts.clamp_min(1))
    sample_weights = class_weights_for_sampling[labels]
    return WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)


def make_loader(
    encoded_split: dict[str, torch.Tensor],
    batch_size: int,
    shuffle: bool = False,
    use_weighted_sampler: bool = False,
) -> DataLoader:
    workers = resolve_dataloader_workers()
    sampler = make_weighted_sampler(encoded_split["labels"]) if use_weighted_sampler else None
    return DataLoader(
        ReviewDataset(encoded_split),
        batch_size=batch_size,
        shuffle=shuffle and sampler is None,
        sampler=sampler,
        num_workers=workers,
        pin_memory=PIN_MEMORY,
        persistent_workers=workers > 0,
    )

### 5.8 DataLoader для каждой архитектуры

Каждая модель получает свой `max_len` и batch size. Это позволяет держать RNN легче, не ухудшая CNN.

In [ ]:
def make_loaders(max_len: int, batch_size: int) -> dict[str, DataLoader]:
    encoded = encoded_by_max_len[max_len]
    use_sampler = BALANCE_STRATEGY == "sampler"
    return {
        "train": make_loader(encoded["train"], batch_size=batch_size, shuffle=True, use_weighted_sampler=use_sampler),
        "val": make_loader(encoded["val"], batch_size=batch_size * 2, shuffle=False),
        "test": make_loader(encoded["test"], batch_size=batch_size * 2, shuffle=False),
    }


loaders_by_model = {
    "TrainableCNN": make_loaders(CNN_MAX_LEN, CNN_BATCH_SIZE),
    "TrainablePackedBiGRU": make_loaders(RNN_MAX_LEN, RNN_BATCH_SIZE),
    "TrainableCNNBiGRU": make_loaders(COMBINED_MAX_LEN, COMBINED_BATCH_SIZE),
}

## Классификация нейронными сетями

Это основная часть лабораторной V. Все модели используют обучаемый `nn.Embedding` поверх токенизированных последовательностей. Эксперименты отличаются тем, как извлекаются признаки из текста: свёртками Conv1D, рекуррентным контекстом GRU или объединением обеих веток.

### 6.1 Pooling helpers

Masked pooling нужен для RNN: он усредняет и максимизирует только реальные токены, не учитывая PAD.

In [ ]:
def sequence_mask(lengths: torch.Tensor, max_len: int) -> torch.Tensor:
    return torch.arange(max_len, device=lengths.device).unsqueeze(0) < lengths.unsqueeze(1)


def masked_mean_pool(outputs: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
    mask = sequence_mask(lengths, outputs.size(1)).unsqueeze(-1).to(outputs.dtype)
    summed = (outputs * mask).sum(dim=1)
    denom = lengths.clamp_min(1).unsqueeze(1).to(outputs.dtype)
    return summed / denom


def masked_max_pool(outputs: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
    mask = sequence_mask(lengths, outputs.size(1)).unsqueeze(-1)
    fill_value = torch.finfo(outputs.dtype).min
    return outputs.masked_fill(~mask, fill_value).max(dim=1).values


class AttentionPool(nn.Module):
    def __init__(self, feature_dim: int):
        super().__init__()
        self.score = nn.Linear(feature_dim, 1)

    def forward(self, outputs: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
        mask = sequence_mask(lengths, outputs.size(1))
        scores = self.score(outputs).squeeze(-1)
        scores = scores.masked_fill(~mask, torch.finfo(scores.dtype).min)
        weights = torch.softmax(scores, dim=1).unsqueeze(-1)
        return (outputs * weights).sum(dim=1)


def count_parameters(model: nn.Module) -> int:
    return sum(param.numel() for param in model.parameters() if param.requires_grad)

### 6.2 Multi-kernel CNN

CNN ловит локальные n-граммы разной длины. `max pooling` сохраняет самый сильный сигнал, а `mean pooling` добавляет информацию о среднем фоне отзыва.

In [ ]:
class TextCNNMulti(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        embedding_dim: int = EMBEDDING_DIM,
        num_classes: int = NUM_CLASSES,
        pad_idx: int = PAD_IDX,
        num_filters: int = 192,
        kernel_sizes: tuple[int, ...] = (1, 2, 3, 4, 5, 7, 9),
        embedding_dropout: float = 0.15,
        dropout: float = 0.45,
    ):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
        self.embedding_dropout = nn.Dropout(embedding_dropout)
        self.convs = nn.ModuleList([
            nn.Conv1d(embedding_dim, num_filters, kernel_size=kernel_size, padding=kernel_size // 2)
            for kernel_size in kernel_sizes
        ])
        feature_dim = len(kernel_sizes) * num_filters * 2
        self.norm = nn.LayerNorm(feature_dim)
        self.classifier = nn.Sequential(
            nn.Linear(feature_dim, 512),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(512, num_classes),
        )

    def forward(self, input_ids: torch.Tensor, lengths: torch.Tensor | None = None) -> torch.Tensor:
        x = self.embedding_dropout(self.embedding(input_ids)).transpose(1, 2)
        features = []
        for conv in self.convs:
            y = F.relu(conv(x))
            features.append(F.adaptive_max_pool1d(y, 1).squeeze(-1))
            features.append(F.adaptive_avg_pool1d(y, 1).squeeze(-1))
        return self.classifier(self.norm(torch.cat(features, dim=1)))

### 6.3 Packed BiGRU

Рекуррентная модель читает последовательность с учетом порядка слов. `pack_padded_sequence` убирает лишнюю работу по PAD-токенам, а финальные признаки собираются из hidden state, mean pooling и max pooling.

In [ ]:
class TextPackedRNN(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        embedding_dim: int = EMBEDDING_DIM,
        hidden_size: int = 256,
        num_classes: int = NUM_CLASSES,
        pad_idx: int = PAD_IDX,
        rnn_type: str = "gru",
        num_layers: int = 1,
        embedding_dropout: float = 0.15,
        dropout: float = 0.45,
    ):
        super().__init__()
        self.rnn_type = rnn_type.lower()
        rnn_cls = nn.GRU if self.rnn_type == "gru" else nn.LSTM
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
        self.embedding_dropout = nn.Dropout(embedding_dropout)
        self.rnn = rnn_cls(
            embedding_dim,
            hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=0.25 if num_layers > 1 else 0.0,
        )
        rnn_out_dim = hidden_size * 2
        self.attention_pool = AttentionPool(rnn_out_dim)
        feature_dim = rnn_out_dim * 4
        self.norm = nn.LayerNorm(feature_dim)
        self.classifier = nn.Sequential(
            nn.Linear(feature_dim, 512),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(512, num_classes),
        )

    def forward(self, input_ids: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
        embedded = self.embedding_dropout(self.embedding(input_ids))
        packed = pack_padded_sequence(
            embedded,
            lengths.detach().cpu(),
            batch_first=True,
            enforce_sorted=False,
        )
        packed_outputs, hidden = self.rnn(packed)
        outputs, _ = pad_packed_sequence(packed_outputs, batch_first=True, total_length=input_ids.size(1))

        if isinstance(hidden, tuple):
            hidden = hidden[0]
        final_hidden = torch.cat([hidden[-2], hidden[-1]], dim=1)
        lengths_on_device = lengths.to(outputs.device)
        features = torch.cat([
            final_hidden,
            masked_mean_pool(outputs, lengths_on_device),
            masked_max_pool(outputs, lengths_on_device),
            self.attention_pool(outputs, lengths_on_device),
        ], dim=1)
        return self.classifier(self.norm(features))

### 6.4 Комбинированная CNN+BiGRU

Комбинированная модель извлекает локальные фразы CNN-веткой и последовательный контекст GRU-веткой, затем объединяет признаки в общий классификатор.

In [ ]:
class TextCNNBiGRU(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        embedding_dim: int = EMBEDDING_DIM,
        num_classes: int = NUM_CLASSES,
        pad_idx: int = PAD_IDX,
        cnn_filters: int = 128,
        kernel_sizes: tuple[int, ...] = (1, 2, 3, 4, 5, 7),
        gru_hidden: int = 192,
        embedding_dropout: float = 0.15,
        dropout: float = 0.5,
    ):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
        self.embedding_dropout = nn.Dropout(embedding_dropout)
        self.convs = nn.ModuleList([
            nn.Conv1d(embedding_dim, cnn_filters, kernel_size=kernel_size, padding=kernel_size // 2)
            for kernel_size in kernel_sizes
        ])
        self.gru = nn.GRU(embedding_dim, gru_hidden, batch_first=True, bidirectional=True)
        rnn_out_dim = gru_hidden * 2
        self.attention_pool = AttentionPool(rnn_out_dim)

        cnn_dim = len(kernel_sizes) * cnn_filters * 2
        rnn_dim = rnn_out_dim * 4
        feature_dim = cnn_dim + rnn_dim
        self.norm = nn.LayerNorm(feature_dim)
        self.classifier = nn.Sequential(
            nn.Linear(feature_dim, 512),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )

    def forward(self, input_ids: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
        embedded = self.embedding_dropout(self.embedding(input_ids))

        cnn_input = embedded.transpose(1, 2)
        cnn_features = []
        for conv in self.convs:
            y = F.relu(conv(cnn_input))
            cnn_features.append(F.adaptive_max_pool1d(y, 1).squeeze(-1))
            cnn_features.append(F.adaptive_avg_pool1d(y, 1).squeeze(-1))

        packed = pack_padded_sequence(embedded, lengths.detach().cpu(), batch_first=True, enforce_sorted=False)
        packed_outputs, hidden = self.gru(packed)
        outputs, _ = pad_packed_sequence(packed_outputs, batch_first=True, total_length=input_ids.size(1))
        lengths_on_device = lengths.to(outputs.device)
        final_hidden = torch.cat([hidden[-2], hidden[-1]], dim=1)
        rnn_features = [
            final_hidden,
            masked_mean_pool(outputs, lengths_on_device),
            masked_max_pool(outputs, lengths_on_device),
            self.attention_pool(outputs, lengths_on_device),
        ]

        features = torch.cat([*cnn_features, *rnn_features], dim=1)
        return self.classifier(self.norm(features))

### 6.5 Конфигурации экспериментов

Три эксперимента покрывают требования задания: только Conv1D, только рекуррентная модель и комбинированная архитектура.

In [ ]:
def build_cnn() -> nn.Module:
    return TextCNNMulti(vocab_size=len(vocab))


def build_rnn() -> nn.Module:
    return TextPackedRNN(vocab_size=len(vocab), rnn_type="gru")


def build_combined() -> nn.Module:
    return TextCNNBiGRU(vocab_size=len(vocab))


model_configs = [
    {
        "name": "TrainableCNN",
        "description": "Embedding + multi-kernel Conv1D + max/mean pooling",
        "factory": build_cnn,
        "loader_key": "TrainableCNN",
        "lr": 8e-4,
        "clip_grad": None,
    },
    {
        "name": "TrainablePackedBiGRU",
        "description": "Embedding + packed bidirectional GRU + final/mean/max/attention pooling",
        "factory": build_rnn,
        "loader_key": "TrainablePackedBiGRU",
        "lr": 6e-4,
        "clip_grad": GRAD_CLIP,
    },
    {
        "name": "TrainableCNNBiGRU",
        "description": "Parallel Conv1D branch + packed BiGRU branch with attention pooling",
        "factory": build_combined,
        "loader_key": "TrainableCNNBiGRU",
        "lr": 5e-4,
        "clip_grad": GRAD_CLIP,
    },
]

pd.DataFrame([{k: v for k, v in cfg.items() if k != "factory"} for cfg in model_configs])

## Обучение и оценка моделей

Структура раздела повторяет логику лабораторной IV: сначала задаём общие функции обучения и метрик, затем запускаем несколько моделей, сравниваем качество, выбираем лучшую модель и строим `classification_report` с матрицей ошибок. Отличие в том, что вместо классических ML-моделей обучаются нейросетевые архитектуры с `Embedding`.

Модели обучаются по `CrossEntropyLoss`. В этой итерации балансировка задается через `BALANCE_STRATEGY`: по умолчанию используется `WeightedRandomSampler`, а веса классов в loss выключены, чтобы не удваивать компенсацию дисбаланса. Early stopping и scheduler ориентируются на validation `macro-F1`, потому что accuracy плохо отражает качество на несбалансированных классах.

### 7.1 Метрики и class weights

Веса классов считаются по train-набору. Используется степень `0.25`, чтобы компенсировать дисбаланс мягко, не заставляя модель механически пере-предсказывать меньшие классы.

In [ ]:
def labels_from_ids(label_ids: np.ndarray | list[int]) -> list[str]:
    return [ID_TO_LABEL[int(label_id)] for label_id in label_ids]


def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    precision, recall, f1_by_class, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=list(range(NUM_CLASSES)),
        zero_division=0,
    )
    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
    }
    for idx, label in ID_TO_LABEL.items():
        metrics[f"{label}_precision"] = precision[idx]
        metrics[f"{label}_recall"] = recall[idx]
        metrics[f"{label}_f1"] = f1_by_class[idx]
    return metrics


class_counts = train_df[TARGET_COL].value_counts().reindex(LABEL_ORDER)
count_ratio = class_counts.sum() / (NUM_CLASSES * class_counts)
loss_weight_values = np.power(count_ratio.values.astype("float32"), CLASS_WEIGHT_ALPHA)

if BALANCE_STRATEGY == "loss_weights":
    class_weights = torch.tensor(loss_weight_values, dtype=torch.float32, device=DEVICE)
else:
    class_weights = None

criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=LABEL_SMOOTHING)

pd.DataFrame({
    "class": LABEL_ORDER,
    "count": class_counts.values,
    "loss_weight_if_enabled": loss_weight_values,
    "active_balance_strategy": BALANCE_STRATEGY,
})

### 7.2 AMP и перенос batch на устройство

Вспомогательные функции скрывают различия между CUDA и CPU/MPS. На CUDA используется `torch.amp`.

In [ ]:
def autocast_context():
    if USE_AMP:
        return torch.amp.autocast(device_type="cuda", enabled=True)
    return nullcontext()


def make_scaler():
    if USE_AMP:
        return torch.amp.GradScaler("cuda", enabled=True)
    return None


def move_batch(batch: dict[str, torch.Tensor]) -> dict[str, torch.Tensor]:
    return {
        key: value.to(DEVICE, non_blocking=PIN_MEMORY)
        for key, value in batch.items()
    }

### 7.3 Один проход обучения или оценки

Функция используется и для train, и для validation. Во время обучения работает AMP, gradient clipping и optimizer step.

In [ ]:
def run_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer | None = None,
    scaler=None,
    clip_grad: float | None = None,
) -> tuple[float, dict[str, float]]:
    is_train = optimizer is not None
    model.train(is_train)
    losses: list[float] = []
    all_preds: list[np.ndarray] = []
    all_labels: list[np.ndarray] = []

    for batch in tqdm(loader, leave=False, desc="train" if is_train else "eval"):
        batch = move_batch(batch)
        if is_train:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(is_train):
            with autocast_context():
                logits = model(batch["input_ids"], batch["lengths"])
                loss = criterion(logits, batch["labels"])

            if is_train:
                if scaler is not None:
                    scaler.scale(loss).backward()
                    if clip_grad is not None:
                        scaler.unscale_(optimizer)
                        nn.utils.clip_grad_norm_(model.parameters(), clip_grad)
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    loss.backward()
                    if clip_grad is not None:
                        nn.utils.clip_grad_norm_(model.parameters(), clip_grad)
                    optimizer.step()

        losses.append(float(loss.detach().cpu()))
        all_preds.append(logits.detach().argmax(dim=1).cpu().numpy())
        all_labels.append(batch["labels"].detach().cpu().numpy())

    y_pred = np.concatenate(all_preds)
    y_true = np.concatenate(all_labels)
    return float(np.mean(losses)), compute_metrics(y_true, y_pred)

### 7.4 Оценка модели

`evaluate_model` возвращает logits, probabilities, predictions, labels и метрики. Bias можно передать после calibration.

In [ ]:
def apply_logit_bias(logits: torch.Tensor, bias: np.ndarray | None = None) -> torch.Tensor:
    if bias is None:
        return logits
    return logits + torch.tensor(bias, dtype=logits.dtype, device=logits.device).view(1, -1)


@torch.inference_mode()
def evaluate_model(model: nn.Module, loader: DataLoader, bias: np.ndarray | None = None) -> dict:
    model.eval()
    logits_list: list[torch.Tensor] = []
    labels_list: list[torch.Tensor] = []

    for batch in tqdm(loader, leave=False, desc="predict"):
        batch = move_batch(batch)
        logits = model(batch["input_ids"], batch["lengths"])
        logits_list.append(logits.detach().cpu())
        labels_list.append(batch["labels"].detach().cpu())

    logits = torch.cat(logits_list)
    labels = torch.cat(labels_list).numpy()
    adjusted_logits = apply_logit_bias(logits, bias)
    probabilities = torch.softmax(adjusted_logits, dim=1).numpy()
    predictions = adjusted_logits.argmax(dim=1).numpy()

    return {
        "logits": logits,
        "probabilities": probabilities,
        "predictions": predictions,
        "labels": labels,
        "metrics": compute_metrics(labels, predictions),
    }

### 7.5 Optimizer и early stopping

Для всех моделей используется `AdamW`. Scheduler уменьшает learning rate, если validation `macro-F1` перестает расти.

In [ ]:
def make_optimizer(model: nn.Module, lr: float) -> torch.optim.Optimizer:
    return torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)


def train_model(model: nn.Module, config: dict, loaders: dict[str, DataLoader], criterion: nn.Module) -> dict:
    model = model.to(DEVICE)
    optimizer = make_optimizer(model, lr=config["lr"])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=2,
        min_lr=1e-5,
    )
    scaler = make_scaler()

    best_state = deepcopy(model.state_dict())
    best_val_macro_f1 = -1.0
    best_epoch = 0
    wait = 0
    history: list[dict] = []
    start_time = time.perf_counter()

    for epoch in range(1, MAX_EPOCHS + 1):
        train_loss, train_metrics = run_epoch(
            model,
            loaders["train"],
            criterion,
            optimizer=optimizer,
            scaler=scaler,
            clip_grad=config.get("clip_grad"),
        )
        val_loss, val_metrics = run_epoch(model, loaders["val"], criterion)
        scheduler.step(val_metrics["macro_f1"])

        row = {
            "model": config["name"],
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "train_macro_f1": train_metrics["macro_f1"],
            "val_macro_f1": val_metrics["macro_f1"],
            "val_accuracy": val_metrics["accuracy"],
            "lr": optimizer.param_groups[0]["lr"],
        }
        history.append(row)
        print(
            f"{config['name']} epoch {epoch}: "
            f"val_macro_f1={val_metrics['macro_f1']:.4f}, val_acc={val_metrics['accuracy']:.4f}"
        )

        if val_metrics["macro_f1"] > best_val_macro_f1:
            best_val_macro_f1 = val_metrics["macro_f1"]
            best_epoch = epoch
            best_state = deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1
            if wait >= PATIENCE:
                print(f"Early stopping at epoch {epoch}")
                break

    train_time = time.perf_counter() - start_time
    model.load_state_dict(best_state)
    return {
        "model": model,
        "history": pd.DataFrame(history),
        "best_epoch": best_epoch,
        "best_val_macro_f1": best_val_macro_f1,
        "train_time": train_time,
        "params": count_parameters(model),
    }

### 7.6 Logit calibration

Bias подбирается только на validation logits. Test не используется для настройки порогов или bias.

In [ ]:
def find_best_logit_bias(
    logits: torch.Tensor,
    labels: np.ndarray,
    grid: np.ndarray | None = None,
) -> tuple[np.ndarray, float]:
    if grid is None:
        grid = np.linspace(-1.2, 1.2, 25)
    best_bias = np.zeros(NUM_CLASSES, dtype="float32")
    best_score = -1.0

    for b_negative in grid:
        for b_neutral in grid:
            bias = np.array([b_negative, b_neutral, 0.0], dtype="float32")
            predictions = apply_logit_bias(logits, bias).argmax(dim=1).numpy()
            score = f1_score(labels, predictions, average="macro", zero_division=0)
            if score > best_score:
                best_score = score
                best_bias = bias

    return best_bias, best_score

### 7.7 Inference для новых текстов

Новые отзывы проходят ту же предобработку, токенизацию, head-tail truncation и calibration, что и test-набор.

In [ ]:
@torch.inference_mode()
def predict_texts(
    model: nn.Module,
    texts: list[str],
    vocab: dict[str, int],
    max_len: int,
    bias: np.ndarray | None = None,
) -> pd.DataFrame:
    cleaned = [preprocess_text(text) for text in texts]
    input_ids, lengths = encode_texts(cleaned, vocab, max_len)
    model.eval()
    logits = model(input_ids.to(DEVICE), lengths.to(DEVICE)).detach().cpu()
    probabilities = torch.softmax(apply_logit_bias(logits, bias), dim=1).numpy()
    predictions = probabilities.argmax(axis=1)

    result = pd.DataFrame({
        "text": texts,
        "cleaned_text": cleaned,
        "predicted_label": labels_from_ids(predictions),
    })
    for idx, label in ID_TO_LABEL.items():
        result[f"p_{label}"] = probabilities[:, idx]
    return result

### 7.8 Обучение одной модели

Функция собирает обучение, raw validation, calibration и calibrated validation в один reproducible-блок.

In [ ]:
def train_and_calibrate(config: dict, seed: int = SEED) -> tuple[dict, dict, pd.DataFrame]:
    set_seed(seed)
    print(f"Training {config['name']} with seed={seed}")
    model = config["factory"]()
    loaders = loaders_by_model[config["loader_key"]]
    train_result = train_model(model, config, loaders, criterion)

    val_raw = evaluate_model(train_result["model"], loaders["val"])
    best_bias, calibrated_val_macro_f1 = find_best_logit_bias(val_raw["logits"], val_raw["labels"])
    val_calibrated = evaluate_model(train_result["model"], loaders["val"], bias=best_bias)

    row = {
        "model": config["name"],
        "seed": seed,
        "best_epoch": train_result["best_epoch"],
        "raw_val_macro_f1": val_raw["metrics"]["macro_f1"],
        "calibrated_val_macro_f1": val_calibrated["metrics"]["macro_f1"],
        "raw_val_accuracy": val_raw["metrics"]["accuracy"],
        "calibrated_val_accuracy": val_calibrated["metrics"]["accuracy"],
        "neutral_f1": val_calibrated["metrics"]["neutral_f1"],
        "train_time": train_result["train_time"],
        "params": train_result["params"],
    }
    bundle = {
        **train_result,
        "config": config,
        "seed": seed,
        "val_raw": val_raw,
        "val_calibrated": val_calibrated,
        "bias": best_bias,
        "calibrated_val_macro_f1": calibrated_val_macro_f1,
        "max_len": {"TrainableCNN": CNN_MAX_LEN, "TrainablePackedBiGRU": RNN_MAX_LEN, "TrainableCNNBiGRU": COMBINED_MAX_LEN}[config["loader_key"]],
    }
    history = train_result["history"].copy()
    history["seed"] = seed
    return bundle, row, history

### 7.9 Запуск трех экспериментов

Основной запуск обучает по одному seed для каждой архитектуры. Seed ensemble оставлен выключенным, чтобы базовый прогон не становился неожиданно долгим.

In [ ]:
trained_models = {}
validation_rows = []
history_frames = []

for config in model_configs:
    bundle, row, history = train_and_calibrate(config, seed=SEED)
    trained_models[config["name"]] = bundle
    validation_rows.append(row)
    history_frames.append(history)

validation_summary_df = pd.DataFrame(validation_rows).sort_values("calibrated_val_macro_f1", ascending=False)
history_df = pd.concat(history_frames, ignore_index=True)
display(validation_summary_df)

### 7.10 Опциональный seed ensemble

Если нужно выжать более стабильный результат, можно включить `RUN_SEED_ENSEMBLE=True` и обучить выбранные архитектуры на нескольких seed. По умолчанию этот блок ничего не запускает.

In [ ]:
ensemble_bundles = {}

if RUN_SEED_ENSEMBLE:
    for config in model_configs:
        seed_bundles = []
        for seed in ENSEMBLE_SEEDS:
            bundle, _, _ = train_and_calibrate(config, seed=seed)
            seed_bundles.append(bundle)
        ensemble_bundles[config["name"]] = seed_bundles
else:
    print("Seed ensemble выключен. Для запуска установите RUN_SEED_ENSEMBLE=True.")

### 7.11 Динамика validation macro-F1

График нужен для проверки, как быстро модели сходятся и где сработал early stopping.

In [ ]:
fig = px.line(
    history_df,
    x="epoch",
    y="val_macro_f1",
    color="model",
    markers=True,
    title="Динамика validation macro-F1",
    labels={"epoch": "Эпоха", "val_macro_f1": "Validation macro-F1"},
)
fig.update_layout(height=460)
fig.show()

### 7.12 Оценка на test

Test используется только после обучения и calibration. Для каждой модели показываются raw и calibrated метрики.

In [ ]:
def collect_test_results(model_name: str, bundle: dict) -> tuple[dict, dict]:
    loader = loaders_by_model[bundle["config"]["loader_key"]]["test"]
    raw = evaluate_model(bundle["model"], loader)
    calibrated = evaluate_model(bundle["model"], loader, bias=bundle["bias"])
    row = {
        "model": model_name,
        "raw_macro_f1": raw["metrics"]["macro_f1"],
        "calibrated_macro_f1": calibrated["metrics"]["macro_f1"],
        "raw_accuracy": raw["metrics"]["accuracy"],
        "calibrated_accuracy": calibrated["metrics"]["accuracy"],
        "weighted_f1": calibrated["metrics"]["weighted_f1"],
        "neutral_f1": calibrated["metrics"]["neutral_f1"],
        "train_time": bundle["train_time"],
        "params": bundle["params"],
    }
    return row, {"raw": raw, "calibrated": calibrated}


test_rows = []
test_predictions = {}
for model_name, bundle in trained_models.items():
    row, predictions = collect_test_results(model_name, bundle)
    test_rows.append(row)
    test_predictions[model_name] = predictions

test_summary_df = pd.DataFrame(test_rows).sort_values("calibrated_macro_f1", ascending=False)
best_model_name = validation_summary_df.iloc[0]["model"]
best_bundle = trained_models[best_model_name]
best_test = test_predictions[best_model_name]["calibrated"]

display(test_summary_df)
print(f"Best by calibrated validation macro-F1: {best_model_name}")

### 7.13 Classification report и confusion matrix

Для лучшей по validation модели показываем подробный отчет и матрицу ошибок на test.

In [ ]:
report_dict = classification_report(
    labels_from_ids(best_test["labels"]),
    labels_from_ids(best_test["predictions"]),
    labels=LABEL_ORDER,
    output_dict=True,
    zero_division=0,
)
report_df = pd.DataFrame(report_dict).T
cm = confusion_matrix(best_test["labels"], best_test["predictions"], labels=list(range(NUM_CLASSES)))

print(classification_report(
    labels_from_ids(best_test["labels"]),
    labels_from_ids(best_test["predictions"]),
    labels=LABEL_ORDER,
    zero_division=0,
))

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=LABEL_ORDER, yticklabels=LABEL_ORDER)
plt.title(f"Confusion matrix: {best_model_name}")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()

## 8. Анализ ошибок

Разбираем конкретные ошибки лучшей модели. Для отзывов Кинопоиска особенно ожидаемы ошибки между `neutral` и `positive`, потому что нейтральные рецензии часто содержат и похвалу, и критику.

### 8.1 Вспомогательные функции анализа

Сокращаем длинные отзывы, чтобы таблица ошибок оставалась читаемой.

In [ ]:
def shorten_text(text: str, max_chars: int = 320) -> str:
    text = re.sub(r"\s+", " ", str(text)).strip()
    if len(text) <= max_chars:
        return text
    return text[: max_chars - 3] + "..."

### 8.2 Примеры ошибок

Показываем 3-5 неверных классификаций с вероятностями классов. По ним удобнее понять, где модель путает тональность.

In [ ]:
analysis_df = test_df.copy()
analysis_df["true_label"] = labels_from_ids(best_test["labels"])
analysis_df["predicted_label"] = labels_from_ids(best_test["predictions"])
analysis_df["confidence"] = best_test["probabilities"].max(axis=1)
for idx, label in ID_TO_LABEL.items():
    analysis_df[f"p_{label}"] = best_test["probabilities"][:, idx]

errors_df = analysis_df[analysis_df["true_label"] != analysis_df["predicted_label"]].copy()
errors_df = errors_df.sort_values("confidence", ascending=False).head(5)
errors_df["review_short"] = errors_df[TEXT_COL].map(shorten_text)

display(errors_df[["true_label", "predicted_label", "confidence", "p_negative", "p_neutral", "p_positive", "review_short"]])

## 9. Проверка на новых текстах

Пять коротких отзывов проходят через тот же pipeline. Рядом указана ожидаемая метка, чтобы можно было быстро оценить поведение модели на простых примерах.

### 9.1 Новые отзывы

Отзывы подобраны так, чтобы покрыть явно положительную, явно отрицательную и смешанную тональность.

In [ ]:
new_examples = [
    {"text": "Фильм получился живым и смешным, актеры отлично держат темп.", "expected_label": "positive"},
    {"text": "Сюжет разваливается уже к середине, финал совсем не работает.", "expected_label": "negative"},
    {"text": "Нормальное кино на вечер: есть удачные сцены, но без сильного впечатления.", "expected_label": "neutral"},
    {"text": "Не шедевр, но персонажи понравились, а музыка прямо вытянула настроение.", "expected_label": "positive"},
    {"text": "Картинка красивая, зато диалоги скучные и слишком затянутые.", "expected_label": "neutral"},
]
new_examples_df = pd.DataFrame(new_examples)
display(new_examples_df)

### 9.2 Предсказания на новых отзывах

Используется лучшая модель и calibration bias, подобранный только на validation.

In [ ]:
new_predictions = predict_texts(
    best_bundle["model"],
    new_examples_df["text"].tolist(),
    vocab,
    max_len=best_bundle["max_len"],
    bias=best_bundle["bias"],
)
new_predictions.insert(1, "expected_label", new_examples_df["expected_label"])
display(new_predictions)

## 10. Выводы

Финальные выводы строятся после выполнения ноутбука: они привязаны к метрикам лучшей модели, структуре ошибок и сравнению архитектур.

### 10.1 Итоговая интерпретация

Эта ячейка формирует связный вывод по фактическим метрикам после запуска всех экспериментов.

In [ ]:
best_val_row = validation_summary_df[validation_summary_df["model"] == best_model_name].iloc[0]
best_test_row = test_summary_df[test_summary_df["model"] == best_model_name].iloc[0]

summary_text = f"""
Лучшей моделью по validation `macro-F1` стала `{best_model_name}`: после calibration она получила `macro-F1={best_val_row['calibrated_val_macro_f1']:.4f}` на validation и `macro-F1={best_test_row['calibrated_macro_f1']:.4f}` на test. Для этой задачи я смотрю именно на `macro-F1`, потому что accuracy может выглядеть приемлемо даже тогда, когда модель плохо распознает менее частые классы.

В этой итерации train, validation и test оставлены в естественном распределении классов, а частотные признаки строятся без утечки: словарь строится только по `train_df`, calibration подбирается только на validation. Для борьбы с дисбалансом по умолчанию используется `WeightedRandomSampler`, а не одновременная комбинация sampler и class weights.

Главные изменения относительно классического pipeline лабораторной IV: TF-IDF заменён на последовательности индексов и обучаемый Embedding, классические классификаторы заменены на Conv1D, GRU и комбинированную CNN+GRU модель, а подбор лучшей модели проводится по validation macro-F1. По матрице ошибок нужно отдельно смотреть пары `neutral ↔ positive` и `neutral ↔ negative`: нейтральные отзывы часто смешивают похвалу и критику, а разметка сводит этот баланс к одной метке. Если качество окажется ограниченным, следующий шаг уже вне базовых ограничений лабораторной — предобученные fastText/ruBERT-подобные представления.
"""
display(Markdown(summary_text))

### 10.2 Что можно улучшить дальше

Основной notebook оставлен строго в рамках задания: обучаемый `Embedding` поверх токенизированных последовательностей, Conv1D, GRU и комбинированная архитектура. Для дальнейшего улучшения качества можно попробовать предобученные эмбеддинги, например fastText-подобные решения для русского языка: они дают более сильные начальные векторы слов и могут особенно помочь редким словоформам и словам вне словаря. Я бы выносил такой эксперимент отдельно, чтобы не смешивать обязательную часть лабораторной с расширенным вариантом.